# Challenge Four: Programming an Agent Workflow
### Agentic AI with the Google Agent Development Kit (ADK) — Skills Validation Workshop (15 Points)

**Objectives:**
- Program a complex process using **Google ADK Workflow Agents** (`SequentialAgent` or `LoopAgent`).
- Implement an answer team that verifies and refines answers prior to returning them to the user:
  1. **Greeter (Root) Agent (`greeter`)**: Welcomes the user, maintains context, and coordinates with the answer team.
  2. **Search Agent (`search_agent`)**: Uses the built-in `google_search` tool to find factual, up-to-date data.
  3. **Critique Agent (`critique_agent`)**: Reviews the initial response for accuracy, completeness, and clarity, providing actionable suggestions.
  4. **Refine Agent (`refine_agent`)**: Rewrites and polishes the response incorporating the critique suggestions.
  5. **Sequential Workflow (`answer_team`)**: Chains `search_agent` -> `critique_agent` -> `refine_agent` into a coordinated pipeline.
- Maintain state using `append_to_state` and inspect streamed workflow events demonstrating each sub-agent in execution.

## 1. Install Dependencies
Install Google ADK, Vertex AI reasoning engines, and supporting packages.

In [1]:
# In Google Cloud Colab Enterprise, install the required packages
!pip install -q --upgrade \
    "google-cloud-aiplatform[agent_engines,adk]" \
    google-adk \
    requests \
    googlemaps \
    litellm

## 2. Google Cloud Authentication & Vertex AI Setup (ADC)
Initialize Vertex AI using Application Default Credentials (ADC) and project configuration.

In [2]:
import os
import logging
from typing import Optional, List, Dict, Any, Tuple
from IPython.display import Markdown, display

import google.auth
import vertexai
from vertexai.preview import reasoning_engines
from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search, ToolContext

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("WorkflowAgents")

# 1. Resolve Application Default Credentials (ADC)
try:
    credentials, project_id = google.auth.default(
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )
    print(f"[Auth] Loaded Application Default Credentials (ADC) for project: {project_id}")
except Exception as e:
    print(f"[Auth Notice] ADC not found ({e}). Falling back to workshop project ID.")
    credentials, project_id = None, os.getenv("GOOGLE_CLOUD_PROJECT")

# Default to workshop credentials project if running in fresh environment
PROJECT_ID = project_id or os.getenv("GOOGLE_CLOUD_PROJECT", "qwiklabs-gcp-01-7a38d08f6f78")
LOCATION = os.getenv("GOOGLE_CLOUD_REGION", "us-central1")

# Instruct Google ADK to route to Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["VERTEXAI_PROJECT"] = PROJECT_ID
os.environ["VERTEXAI_LOCATION"] = LOCATION

vertexai.init(project=PROJECT_ID, location=LOCATION, credentials=credentials)
print(f"[Vertex AI] Initialized in {LOCATION} for project {PROJECT_ID}.")

[Auth] Loaded Application Default Credentials (ADC) for project: qwiklabs-gcp-01-7a38d08f6f78
[Vertex AI] Initialized in us-central1 for project qwiklabs-gcp-01-7a38d08f6f78.


## 3. Maintaining State across Workflow Steps
Per Slide 55, define `append_to_state` with `tool_context: ToolContext` to track intermediate steps and maintain long-running context.

In [3]:
def append_to_state(tool_context: ToolContext, field: str, response: str) -> Dict[str, Any]:
    """
    Appends an agent response or milestone to a specific field in the shared workflow state.

    Args:
        tool_context (ToolContext): Injected ADK context holding the session state.
        field (str): Name of the state field (e.g., 'search_draft', 'critique_notes', 'final_answer').
        response (str): Text content to append.

    Returns:
        Dict[str, Any]: Status dictionary indicating successful update.
    """
    existing_state = tool_context.state.get(field, [])
    if isinstance(existing_state, list):
        tool_context.state[field] = existing_state + [response]
    else:
        tool_context.state[field] = [existing_state, response]
    return {"status": "success", "field": field, "count": len(tool_context.state[field])}

print("[Tool Registered] append_to_state ready for workflow state tracking.")

[Tool Registered] append_to_state ready for workflow state tracking.


## 4. Define Specialized Sub-Agents: Search, Critique, and Refine
Per Challenge 4 Instructions:
- **Search Agent**: Finds factual data to answer the user question using `google_search`.
- **Critique Agent**: Evaluates the initial response and makes recommendations for completeness, accuracy, and clarity.
- **Refine Agent**: Rewrites and polishes the final answer incorporating all critique suggestions.

In [4]:
MODEL_NAME = "gemini-2.5-flash"

# 1. Search Agent
SEARCH_INSTRUCTIONS = """You are the Search Agent in the answer team.
Your objective is to gather raw, factual, up-to-date information from the internet to answer the user's question.

Guidelines:
1. Formulate search queries using the `google_search` tool.
2. Retrieve key facts, statistics, dates, and background context.
3. Provide a factual initial draft answering the query clearly.
4. Present the facts clearly so the Critique Agent can review them thoroughly.
"""

search_agent = Agent(
    name="search_agent",
    model=MODEL_NAME,
    description="Finds raw factual data to answer the user's question using Google Search.",
    instruction=SEARCH_INSTRUCTIONS,
    tools=[google_search],
)

# 2. Critique Agent
CRITIQUE_INSTRUCTIONS = """You are the Critique Agent in the answer team.
Your objective is to rigorously inspect and evaluate the initial draft generated by the Search Agent.

Guidelines:
1. Evaluate the information for:
   - **Completeness**: Are there key facts, context, or perspectives missing?
   - **Accuracy & Consistency**: Do the numbers, dates, and claims make sense?
   - **Clarity & Structure**: Is the explanation concise, well-structured, and easy to understand?
2. Provide constructive, specific suggestions and bulleted recommendations for improvement.
3. Point out any ambiguities or technical jargon that needs clearer explanation.
4. Do NOT write the final answer yourself; your job is to guide the Refine Agent.
"""

critique_agent = Agent(
    name="critique_agent",
    model=MODEL_NAME,
    description="Reviews the initial response and suggests improvements for accuracy, completeness, and clarity.",
    instruction=CRITIQUE_INSTRUCTIONS,
)

# 3. Refine Agent
REFINE_INSTRUCTIONS = """You are the Refine Agent in the answer team.
Your objective is to craft the final, polished response for the user by integrating the initial findings and the critique suggestions.

Guidelines:
1. Review both the Search Agent's factual draft and the Critique Agent's recommendations.
2. Rewrite the response incorporating all suggested improvements.
3. Format the final output professionally using clear Markdown headings, bullet points, and source citations where applicable.
4. Ensure the final response directly and thoroughly answers the original user question with maximum clarity and authority.
"""

refine_agent = Agent(
    name="refine_agent",
    model=MODEL_NAME,
    description="Rewrites the response based on the critique to produce a polished final answer.",
    instruction=REFINE_INSTRUCTIONS,
)

print(f"[Specialized Sub-Agents Initialized]")
print(f"  1. {search_agent.name} (Tools: {[type(t).__name__ for t in search_agent.tools]})")
print(f"  2. {critique_agent.name}")
print(f"  3. {refine_agent.name}")

[Specialized Sub-Agents Initialized]
  1. search_agent (Tools: ['GoogleSearchTool'])
  2. critique_agent
  3. refine_agent


## 5. Build the Sequential Workflow Agent: Answer Team
Combine `search_agent`, `critique_agent`, and `refine_agent` into a `SequentialAgent`.
Each agent in the sequence completes its task before passing the enhanced context to the next agent.

In [5]:
answer_team = SequentialAgent(
    name="answer_team",
    description="Answers questions by verifying and refining information sequentially (Search -> Critique -> Refine).",
    sub_agents=[search_agent, critique_agent, refine_agent],
)

print(f"[Workflow Agent Configured] {answer_team.name}")
print(f"  --> Sequential Steps: {' -> '.join([sa.name for sa in answer_team.sub_agents])}")

[Workflow Agent Configured] answer_team
  --> Sequential Steps: search_agent -> critique_agent -> refine_agent


/tmp/ipykernel_58505/972880929.py:1: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  answer_team = SequentialAgent(


## 6. Build the Greeter (Root Coordinator) Agent
Create the primary entry point `greeter` that welcomes users, coordinates context, and delegates to `answer_team`.

In [6]:
GREETER_INSTRUCTIONS = """You are the primary Greeter and Workflow Coordinator Agent.
Your role is to welcome the user, understand their query, and coordinate with the `answer_team` sequential workflow.

Workflow:
1. When a user asks a factual, technical, or research question, delegate it immediately to the `answer_team`.
2. The `answer_team` executes a 3-step verification and refinement pipeline:
   - Search: Gathers relevant facts using `google_search`.
   - Critique: Identifies missing elements, checks accuracy, and suggests improvements.
   - Refine: Produces the final polished response.
3. Once the `answer_team` finishes, present the refined, authoritative answer clearly to the user.
"""

greeter_agent = Agent(
    name="greeter",
    model=MODEL_NAME,
    description="Primary entry point that coordinates the answer team workflow.",
    instruction=GREETER_INSTRUCTIONS,
    tools=[append_to_state],
    sub_agents=[answer_team],
)

print(f"[Root Agent Initialized] {greeter_agent.name}")
print(f"  --> Sub-agents: {[sa.name for sa in greeter_agent.sub_agents]}")
print(f"  --> Tools: {[t.__name__ for t in greeter_agent.tools]}")

[Root Agent Initialized] greeter
  --> Sub-agents: ['answer_team']
  --> Tools: ['append_to_state']


## 7. Initialize Vertex AI AdkApp & Session
Host the workflow within Vertex AI `reasoning_engines.AdkApp` and create a persistent session.

In [7]:
# Wrap into Vertex AI AdkApp
app = reasoning_engines.AdkApp(agent=greeter_agent)

# Create user session
user_id = "workshop-student-ch4"
session = app.create_session(user_id=user_id)
session_id = session["id"] if isinstance(session, dict) else session.id
print(f"[Session Ready] User: {user_id} | Session ID: {session_id}")

/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:1138: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


[Session Ready] User: workshop-student-ch4 | Session ID: 7cb3f87a-581d-4c84-8d90-6187b2fbd4a2


## 8. Test the Agent Workflow & Output Event Streams
Per Challenge 4, Step 4: *"Write code to test your agents. Output events to your notebook that demonstrate the use of the sub agents."*

We test the workflow with research queries to observe:
1. Invocation of `greeter` -> Delegation to `answer_team`
2. Sequential step 1: `search_agent` gathering data
3. Sequential step 2: `critique_agent` evaluating and suggesting enhancements
4. Sequential step 3: `refine_agent` synthesizing the final polished response

In [8]:
def test_workflow_stream(query: str, test_title: str):
    print("=" * 80)
    print(f"TEST: {test_title}")
    print(f"QUERY: '{query}'")
    print("=" * 80)

    invoked_steps = []
    full_response = ""

    try:
        for event in app.stream_query(user_id=user_id, session_id=session_id, message=query):
            author = event.get("author") or event.get("node_info", {}).get("path")
            if author and str(author) not in invoked_steps:
                invoked_steps.append(str(author))
                print(f"  [WORKFLOW STEP EVENT] Node / Agent: {author}")

            if "content" in event and event["content"].get("parts"):
                for part in event["content"]["parts"]:
                    if "text" in part:
                        full_response += part["text"]

        print(f"\n--> Completed Workflow Execution Path: {invoked_steps}")
        print("\n--- FINAL REFINED RESPONSE ---")
        display(Markdown(full_response if full_response else "(Workflow execution completed)"))

    except Exception as e:
        if "DefaultCredentialsError" in str(e) or "GoogleAuthError" in str(e):
            print(f"[Notice] In offline / local mode without ADC: {e}")
            print("  --> In Google Cloud Colab Enterprise, ADC authenticates automatically.")
        else:
            print(f"[Query Error]: {e}")


# Test 1: Technological innovation query
test_workflow_stream(
    query="What are the latest developments and timeline for the ITER nuclear fusion experiment?",
    test_title="1. NUCLEAR FUSION VERIFIED & REFINED ANSWER"
)

# Test 2: Space exploration & planetary science query
test_workflow_stream(
    query="What did NASA's Europa Clipper mission discover or aim to accomplish on Jupiter's moon Europa?",
    test_title="2. EUROPA CLIPPER VERIFIED & REFINED ANSWER"
)

TEST: 1. NUCLEAR FUSION VERIFIED & REFINED ANSWER
QUERY: 'What are the latest developments and timeline for the ITER nuclear fusion experiment?'
  [WORKFLOW STEP EVENT] Node / Agent: greeter
  [WORKFLOW STEP EVENT] Node / Agent: search_agent
  [WORKFLOW STEP EVENT] Node / Agent: critique_agent
  [WORKFLOW STEP EVENT] Node / Agent: refine_agent

--> Completed Workflow Execution Path: ['greeter', 'search_agent', 'critique_agent', 'refine_agent']

--- FINAL REFINED RESPONSE ---


The International Thermonuclear Experimental Reactor (ITER) project, a global collaboration aimed at demonstrating the feasibility of fusion power, has seen significant developments and a revised timeline. The project, located in Cadarache, southern France, is currently undergoing assembly with a new schedule, known as Baseline 2024, put forth in July 2024.

**Revised Timeline:**
The updated timeline for the ITER project pushes key operational milestones back by several years compared to previous schedules:
*   **Start of Research Operation (with deuterium-deuterium fusion):** This initial phase of operations, which will include full magnetic energy and plasma current, is now targeted for 2034 or 2035. The previous target for "first plasma" was 2025.
*   **Full Magnetic Energy and Plasma Current Operation (deuterium-deuterium plasmas):** This is expected in 2036.
*   **Deuterium-Tritium Operation (full fusion power, Q=10):** The project aims to begin operations with the relevant fusion fuel (deuterium-tritium) in 2039. This phase will target a Q-value of 10, meaning it is designed to produce ten times more thermal power than it consumes.

The delays are attributed to factors such as the COVID-19 pandemic, quality control issues, and an overly optimistic initial planning for such a complex, first-of-its-kind endeavor. The new baseline is intended to ensure a more scientifically and technically robust initial operational phase, including the installation of components required for operation at 15 megaamperes, which were not part of the previous low-energy machine-test phase. The rebaseline is also estimated to add approximately €5 billion to the project's cost, bringing the total estimated budget to between €18 billion and €20 billion, up from an original €5 billion.

**Latest Developments:**
As of late 2026, the ITER project is making substantial progress in its assembly phase:
*   **Component Assembly:** By July 2026, six out of nine vacuum vessel sector modules had been successfully installed in the tokamak pit, signifying that two-thirds of the tokamak's core is now in place. The ninth and final vacuum vessel segment, sector #2, was delivered to the site in October 2026, meaning all major components for the "core machine" (including vacuum vessel, superconducting magnets, and cryostat) are now on site. Assembly teams have also begun work on vacuum vessel sector 3 and initiated preparations for welding the vacuum vessel. The final module is anticipated to be installed by mid-2027.
*   **Thermal Shield Repair:** A three-year repair project on the vacuum vessel thermal shield, addressing microscopic stress-corrosion cracks discovered in 2021, was completed in July 2026.
*   **Magnet Testing:** In September 2026, preparations commenced for testing a 330-metric ton toroidal field coil at 34 kA. Further upgrades are planned before initiating 68 kA testing with toroidal field coil #19.
*   **Material Change:** The plasma-facing blanket first wall material is being switched from beryllium to tungsten.
*   **ITER Council Endorsement:** In November 2024, the ITER Council reviewed the project's progress and endorsed the Proposed Baseline 2024, confirming that the project is performing according to the schedule of this new baseline.

The ITER project is a collaborative effort involving China, the European Union, India, Japan, South Korea, the Russian Federation, and the United States, aiming to establish the scientific and technological basis for future fusion power plants. It is designed to demonstrate fusion power but will not generate electricity itself.The draft provides a good overview of the ITER project's revised timeline and lists several developments. However, there are significant issues with the temporal accuracy of the "Latest Developments" section, and some technical jargon could be clarified.

Here are specific recommendations for improvement:

*   **Critical Accuracy Issue: Temporal Mismatch in "Latest Developments"**: The most pressing issue is that the section "As of late 2026" describes future events (July 2026, October 2026, September 2026) as if they have already occurred. The request is for "latest developments," which implies recent, actual occurrences (e.g., Q1/Q2 2024). This section needs to be corrected to reflect actual current progress (mid-2024) or, if these are indeed *projected* milestones under the new Baseline 2024, they must be explicitly stated as *planned future developments* rather than accomplished facts.
    *   **Recommendation:** Re-evaluate the source data for actual 2024 developments. If the information about 2026 comes from a forward-looking plan under Baseline 2024, rephrase the section to clearly state these are "Projected Milestones under Baseline 2024" or "Key Planned Deliverables by [Year]" and ensure the phrasing indicates future intent (e.g., "is planned to be installed," "preparations are expected to commence").

*   **Clarify "Baseline 2024" Scope**: While it mentions the cost increase and robust operational phase, a slightly more explicit statement about "Baseline 2024" as a comprehensive re-evaluation of the project's technical scope, schedule, and cost would enhance understanding.
    *   **Recommendation:** Briefly explain that "Baseline 2024" is a revised, comprehensive project plan that addresses past challenges and sets new, more realistic targets for construction, assembly, and operation, reflecting updated technical requirements and a thorough risk assessment.

*   **Explain Technical Jargon for Broader Accessibility**: Several terms are used that might not be clear to a general audience.
    *   **Recommendation:**
        *   Briefly define "tokamak" (e.g., "a toroidal magnetic confinement device").
        *   Explain "vacuum vessel sector modules" (e.g., "large segments that form the donut-shaped chamber where fusion will occur").
        *   Clarify "toroidal field coil" (e.g., "powerful magnets that create the magnetic field to confine the plasma").
        *   Briefly note what "34 kA" and "68 kA" refer to (e.g., "kiloamperes, indicating the strength of the electrical current used for the magnetic fields").
        *   Explain the purpose of the "plasma-facing blanket first wall" (e.g., "the inner lining of the vacuum vessel that directly faces the superheated plasma").

*   **Distinguish Between "First Plasma" and "Start of Research Operation" More Clearly**: The draft mentions "First Plasma" as the previous target and "Start of Research Operation" as the new one. While the distinction is implicit, making it more explicit could prevent confusion.
    *   **Recommendation:** Clearly state that "First Plasma" was the initial, less demanding milestone, whereas "Start of Research Operation" under Baseline 2024 signifies a more substantial initial operational phase, including higher magnetic energy and plasma current.

*   **Add a Concluding Sentence**: The draft ends abruptly after listing the material change and council endorsement. A brief concluding remark would provide a more polished finish.
    *   **Recommendation:** Add a sentence summarizing the overall trajectory or significance, e.g., "Despite the extended timeline, these developments reflect ITER's ongoing progress and the international commitment to realizing fusion energy."The International Thermonuclear Experimental Reactor (ITER) project, a monumental global collaboration aimed at demonstrating the scientific and technological feasibility of fusion power, has recently undergone a significant re-evaluation of its schedule and scope. Located in Cadarache, southern France, the project is currently in its complex assembly phase, operating under a newly established plan known as Baseline 2024.

### Revised Timeline: Baseline 2024

The updated timeline, officially unveiled in July 2024 and endorsed by the ITER Council in November 2024, represents a comprehensive re-evaluation of the project's technical scope, schedule, and cost. This Baseline 2024 addresses past challenges, incorporates updated technical requirements, and sets more realistic targets for construction, assembly, and operation, following a thorough risk assessment. It pushes key operational milestones back by several years compared to previous schedules.

**Key Milestones under Baseline 2024:**

*   **Start of Research Operation (with deuterium-deuterium fusion):** This initial phase, which will involve full magnetic energy and plasma current, is now targeted for **2034 or 2035**. This represents a more substantial and demanding initial operational phase compared to the previously targeted "First Plasma," which was a less stringent milestone for plasma generation.
*   **Full Magnetic Energy and Plasma Current Operation (deuterium-deuterium plasmas):** This is expected in **2036**.
*   **Deuterium-Tritium Operation (full fusion power, Q=10):** The project aims to begin operations with the relevant fusion fuel (deuterium-tritium) in **2039**. This phase will target a Q-value of 10, meaning it is designed to produce ten times more thermal power than it consumes from external heating.

**Context for Delays and Costs:**

The delays are attributed to a confluence of factors, including the global COVID-19 pandemic, challenges related to quality control, and an acknowledgement that initial planning for such a complex, first-of-its-kind endeavor was overly optimistic. The new baseline is designed to ensure a more scientifically and technically robust initial operational phase, incorporating, for example, the installation of components required for operation at 15 megaamperes, which were not part of the previous low-energy machine-test phase. The rebaselining is also estimated to add approximately €5 billion to the project's cost, bringing the total estimated budget to between €18 billion and €20 billion, significantly higher than the original €5 billion estimate.

### Latest Developments & Future Outlook under Baseline 2024

While the new Baseline 2024 sets future operational targets, the project continues to advance with both recent significant endorsements and planned milestones for its complex assembly.

*   **ITER Council Endorsement (Actual):** In **November 2024**, the ITER Council formally reviewed the project's progress and endorsed the Proposed Baseline 2024, confirming the project is proceeding in line with this updated plan.
*   **Material Change (Ongoing):** The plasma-facing blanket first wall material – the inner lining of the vacuum vessel that directly faces the superheated plasma – is being switched from beryllium to tungsten to enhance performance and durability.

**Projected Assembly Milestones (under Baseline 2024):**

*   **Component Assembly:** By **July 2026**, six out of nine vacuum vessel sector modules – large, curved segments that form the donut-shaped chamber (**tokamak**) where fusion will occur – are planned to be successfully installed in the tokamak pit. The ninth and final vacuum vessel segment, sector #2, is projected for delivery to the site by **October 2026**, ensuring all major components for the "core machine" (including vacuum vessel, superconducting magnets, and cryostat) are on site. Assembly teams are also working on vacuum vessel sector 3 and initiating preparations for welding the vacuum vessel. The installation of the final module is anticipated by **mid-2027**.
*   **Thermal Shield Repair:** A three-year repair project on the vacuum vessel thermal shield, addressing microscopic stress-corrosion cracks discovered in 2021, is projected to be completed in **July 2026**.
*   **Magnet Testing:** In **September 2026**, preparations are expected to commence for testing a 330-metric ton toroidal field coil – one of the powerful magnets that create the magnetic field to confine the plasma – at 34 kA (kiloamperes, indicating the strength of the electrical current used for the magnetic fields). Further upgrades are planned before initiating 68 kA testing with toroidal field coil #19.

The ITER project is a collaborative effort involving China, the European Union, India, Japan, South Korea, the Russian Federation, and the United States, aiming to establish the scientific and technological basis for future fusion power plants. It is designed to demonstrate fusion power but will not generate electricity itself. Despite the extended timeline, these developments reflect ITER's ongoing progress and the international commitment to realizing fusion energy.

TEST: 2. EUROPA CLIPPER VERIFIED & REFINED ANSWER
QUERY: 'What did NASA's Europa Clipper mission discover or aim to accomplish on Jupiter's moon Europa?'
  [WORKFLOW STEP EVENT] Node / Agent: greeter
  [WORKFLOW STEP EVENT] Node / Agent: search_agent
  [WORKFLOW STEP EVENT] Node / Agent: critique_agent
  [WORKFLOW STEP EVENT] Node / Agent: refine_agent

--> Completed Workflow Execution Path: ['greeter', 'search_agent', 'critique_agent', 'refine_agent']

--- FINAL REFINED RESPONSE ---


NASA's Europa Clipper mission, launched on October 14, 2024, aims to determine whether Jupiter's icy moon Europa possesses conditions suitable for life. The mission is not designed to detect life itself, but rather to assess Europa's habitability by investigating its potential for liquid water, essential chemistry, and energy sources.

**Key Objectives and Accomplishments of the Mission:**

The Europa Clipper will conduct a detailed study of the moon through approximately 49 close flybys, some as low as 25 kilometers (16 miles) above Europa's surface, after its arrival in the Jupiter system in April 2030.

Its primary objectives include:
*   **Investigating the Ice Shell and Subsurface Ocean:** The mission seeks to understand the nature of Europa's ice shell and definitively confirm the existence and characteristics of the vast saltwater ocean believed to lie beneath it. Scientists estimate this ocean could contain more than twice the amount of water found in all of Earth's oceans combined. The mission will use an ice-penetrating radar to search for subsurface water and measure the ice shell's thickness.
*   **Analyzing Composition:** Europa Clipper will analyze the chemical composition of Europa's surface, its thin atmosphere, and the potential ocean to identify key compounds and their links to the ocean's composition. Spectrometers aboard the spacecraft will help determine the chemical composition.
*   **Studying Geology and Geothermal Activity:** The mission will perform geological measurements, map over 80% of Europa's surface at high resolutions, and search for evidence of recent tectonic activity, resurfacing, or areas where plumes of water vapor might be venting. A thermal instrument will also pinpoint locations of warmer ice and potential water eruptions.
*   **Assessing Habitability:** By gathering these detailed measurements, Europa Clipper will help scientists better understand the astrobiological potential for habitable worlds beyond Earth. While not a life-detection mission, it will determine what ingredients for life are present and how they interact to create a habitable environment.

**Mission Timeline:**
*   **Launch:** The mission successfully launched on October 14, 2024, aboard a SpaceX Falcon Heavy rocket.
*   **Gravity Assists:** The spacecraft will perform gravity assist maneuvers, first with Mars on March 1, 2025, and then with Earth on December 3, 2026, to gain speed for its journey.
*   **Arrival at Jupiter:** Europa Clipper is expected to arrive at the Jupiter system in April 2030, after traveling approximately 2.9 billion kilometers (1.8 billion miles).
*   **Flybys of Europa:** The first of its 49 planned flybys of Europa will commence in March 2031.
*   **Mission End:** The prime mission is planned to conclude in June 2034, with the spacecraft eventually directed to impact Ganymede in September 2034.

To endure Jupiter's harsh radiation environment, the spacecraft's payload and electronics are protected within a thick-walled radiation vault made of titanium and aluminum. The Europa Clipper is the largest spacecraft developed by NASA for an interplanetary mission, spanning 30.5 meters (100 feet) with its solar arrays deployed.The draft provides a comprehensive and well-structured overview of NASA's Europa Clipper mission, detailing its objectives, planned investigations, and timeline. It clearly distinguishes between assessing habitability and directly detecting life, which is crucial for this mission.

Here are some specific suggestions for improvement:

*   **Refine "Key Objectives and Accomplishments" Heading:** The term "Accomplishments" is not accurate as the mission has only recently launched (or is about to launch, depending on the current date relative to October 2024) and its scientific investigations are yet to begin.
    *   **Recommendation:** Change the heading to "Key Objectives and Planned Investigations" or "Key Objectives and Scientific Aims" to accurately reflect the forward-looking nature of these points.

*   **Enhance "Why not detect life?" Explanation:** While the draft states the mission is "not designed to detect life itself," it could briefly elaborate on *why*.
    *   **Recommendation:** Add a sentence or phrase explaining that the mission lacks the specialized instruments (e.g., mass spectrometers optimized for complex organic molecules, microscopes) required to directly identify biosignatures, focusing instead on measuring the environmental conditions (water, chemistry, energy) necessary for life.

*   **Emphasize the Significance of Plumes:** The mention of "plumes of water vapor" is present but could be given slightly more prominence due to their scientific importance.
    *   **Recommendation:** Briefly highlight that if detected and sampled, these plumes could offer a rare opportunity to study the subsurface ocean's composition directly, without needing to drill through Europa's thick ice shell.

*   **Minor Wording for "Definitively Confirm":** In the "Investigating the Ice Shell and Subsurface Ocean" objective, "definitively confirm the existence" could be slightly softened or clarified to reflect the scientific process of building evidence.
    *   **Recommendation:** Change "definitively confirm the existence" to "further investigate and characterize the existence and properties" or "seek to confirm and characterize the existence." While confirmation is a goal, scientific inquiry often involves robust characterization to establish certainty.NASA's Europa Clipper mission, launched on October 14, 2024, is a dedicated effort to investigate Jupiter's enigmatic moon Europa. Its primary goal is to determine whether Europa possesses the necessary conditions to harbor life. While not designed to detect life itself, the mission will thoroughly assess Europa's habitability by studying its potential for liquid water, essential chemical building blocks, and energy sources.

### Key Objectives and Planned Investigations

The Europa Clipper spacecraft will embark on a comprehensive study of Europa through approximately 49 close flybys, some as low as 25 kilometers (16 miles) above the moon's surface. These investigations will commence after its arrival in the Jupiter system in April 2030.

The mission's main scientific objectives include:

*   **Investigating the Ice Shell and Subsurface Ocean:** The mission aims to further investigate and characterize the existence and properties of Europa's vast saltwater ocean, believed to be hidden beneath its icy shell. Scientists estimate this ocean could contain more than twice the amount of water found in all of Earth's oceans combined. The Clipper will utilize an ice-penetrating radar to search for subsurface water and measure the ice shell's thickness.
*   **Analyzing Composition:** Europa Clipper will analyze the chemical composition of Europa's surface, its extremely thin atmosphere (exosphere), and the potential subsurface ocean. This analysis seeks to identify key compounds, including organic molecules, and understand their relationship to the ocean's composition. Spectrometers aboard the spacecraft will aid in determining these chemical signatures.
*   **Studying Geology and Geothermal Activity:** The mission will conduct detailed geological measurements, mapping over 80% of Europa's surface at high resolutions. It will search for evidence of recent tectonic activity, surface resurfacing processes, or areas where plumes of water vapor might be venting into space. These plumes, if confirmed and sampled, could offer a rare opportunity to directly study the subsurface ocean's composition without needing to drill through the thick ice shell. A thermal instrument will also pinpoint locations of warmer ice and potential water eruptions.
*   **Assessing Habitability:** By gathering these detailed measurements across various scientific disciplines, Europa Clipper will significantly advance scientists' understanding of Europa's astrobiological potential. While the mission is not equipped with specialized instruments (such as mass spectrometers optimized for complex organic molecules or microscopes) to directly identify biosignatures or detect life, it will precisely determine what ingredients for life are present and how they interact to create a potentially habitable environment.

### Mission Timeline

*   **Launch:** The mission successfully launched on **October 14, 2024**, aboard a SpaceX Falcon Heavy rocket from NASA's Kennedy Space Center.
*   **Gravity Assists:** The spacecraft will perform gravity assist maneuvers to gain speed for its journey, first with Mars on **March 1, 2025**, and then with Earth on **December 3, 2026**.
*   **Arrival at Jupiter:** Europa Clipper is expected to arrive at the Jupiter system in **April 2030**, after traveling approximately 2.9 billion kilometers (1.8 billion miles).
*   **Flybys of Europa:** The first of its 49 planned scientific flybys of Europa will commence in **March 2031**.
*   **Mission End:** The prime mission is planned to conclude in **June 2034**, with the spacecraft eventually directed to impact Ganymede in **September 2034** to avoid contaminating Europa.

To endure Jupiter's extremely harsh radiation environment, the spacecraft's sensitive payload and electronics are protected within a thick-walled radiation vault made of titanium and aluminum. With its solar arrays deployed, the Europa Clipper is an exceptionally large spacecraft, spanning 30.5 meters (100 feet), representing the largest spacecraft developed by NASA for an interplanetary mission.

## 9. Summary & Verification
✅ **Greeter (Root) Agent**: Coordinates entry point and delegates to `answer_team`.
✅ **Search Agent**: Built-in `google_search` tool retrieves factual data.
✅ **Critique Agent**: Evaluates accuracy, gaps, and structural clarity.
✅ **Refine Agent**: Produces the comprehensive, authoritative final response.
✅ **SequentialAgent Pipeline**: Orchestrates Search -> Critique -> Refine seamlessly.
✅ **State Tracking**: `append_to_state` with `tool_context: ToolContext` enables persistent workflow state.